## Executive summary

A sensible water tank can serve two very different roles:  
(a) an **inertial storage / buffer tank** used to stabilize flow rates, reduce ON/OFF cycling, and support defrosting;  
(b) a **thermal store / TES** used for intentional storage, such as load shifting, peak shaving, PV integration, or backup operation.  
Confusing these two roles almost always leads to over- or under-sizing.

Rules of thumb based on heat pump power are typically intended for case (a). The European standard **EN 15450** (design guideline for heat pump systems) reports a guideline value for buffer volume of **12–35 L per kW** of maximum heat pump capacity. In the literature, the German guideline **VDI 4645** is often referenced with intervals of **20–40 L/kW** depending on the application and assumptions. In the Italian context, one industrial manufacturer indicates **10–20 L/kW** as an indicative criterion and also proposes an **energy-based approach** based on minimum operating time and allowable temperature difference.

By contrast, when the objective is case (b), namely load shifting or peak shaving, the relevant quantity is the **useful stored energy (kWh)**, so the scaling changes quickly. The thermal capacity of water is about **1.16 kWh per m³·K** (equivalently, **0.00116 kWh per L·K**), consistent with the expressions reported in EN 15450 for the energy stored in a water volume. As a consequence, buffer tanks in the **100–300 L** range, while useful for anti-short-cycling purposes, often provide storage durations on the order of only a few minutes if used as TES under loads of several kW.

The choice of usable temperature lift **ΔT** is a crucial multiplier. In recent studies on optimal sizing and control, storage ΔT values range approximately from a few kelvin up to about **20 K**, depending on the use case and the control strategy. Increasing temperatures and ΔT improves TES energy density, but tends to worsen heat pump COP: in one example for district heating conditions, at an outdoor temperature of **5°C**, COP decreases from operating conditions **30/35°C** to **50/55°C**.

If TES is used in **MPC**, stratification (**see next section**) becomes a key issue. **One-node models** (perfectly mixed tanks) are often overly optimistic. Experimental work shows that a **three-node model**, capable of representing stratification, can significantly improve load-shifting performance compared to a one-node model, while reducing plant-model mismatch. **Control-oriented** multi-node tank models are therefore widely studied precisely because they keep complexity compatible with predictive control.

Finally, if the tank is involved in **DHW**, hygienic constraints must be considered. Italian guidance reports that hot water networks maintained above **50°C** are less frequently colonized, and that above **60°C** legionella activity is inhibited as a function of time. This leads in practice to accumulation temperatures around **55–60°C** with downstream mixing valves. **EN 15450** also requires that the system be able to reach **60°C daily, if required**.


## Stratification in TES

Thermal stratification in a sensible water tank refers to the formation of vertical layers at different temperatures, typically with hotter water in the upper region and colder water in the lower region. This phenomenon is important because it determines how much of the stored energy is actually usable during charging and discharging.

For modelling purposes, a perfectly mixed tank assumes a uniform temperature throughout the whole volume. This approximation is simple and computationally attractive, but it may overestimate the effective flexibility of the storage system. In contrast, stratified formulations preserve at least part of the vertical temperature gradient and therefore provide a more realistic representation of the available thermal energy.

In control-oriented applications, multi-node models are often used as a compromise between physical realism and computational tractability. Instead of resolving the full fluid dynamics inside the tank, the storage volume is divided into a small number of layers, each associated with its own temperature state. This allows the model to capture the main effects of stratification while remaining suitable for MPC.

<div style="text-align: center;">
  <img src="Alyphib_Figs/multinode.png" alt="Multi-node TES representation" width="420">
</div>

The figure below highlights the difference between idealized perfect stratification, experimentally observed stratification with a thermocline region, and the limiting case of complete mixing. For MPC-oriented modelling, the objective is usually not to reproduce every internal phenomenon in detail, but to retain enough structure to describe the evolution of useful energy within the tank.

<div style="text-align: center;">
  <img src="Alyphib_Figs/strastification.png" alt="Stratification concepts" width="560">
</div>


## Sizing logic: from building thermal demand to HP and TES models

The sizing-oriented code snippets presented in the following sections are intended to be **adaptive** rather than tied to a single fixed case study. The underlying idea is that the building-side thermal demand is not imposed here a priori: it will depend on the thermal characteristics of the building model, which are defined separately.

In practice, the building model will determine the required thermal load profile and, in particular, the relevant design or peak demand. Once this quantity is available, the heat pump and thermal storage models can be parameterized accordingly. In this sense, the HP and TES models proposed here should be interpreted as **scalable components**, whose main parameters are updated consistently with the thermal request coming from the building side.

This workflow is especially useful when detailed design data are not yet available. At an early stage, one may rely on simplified load estimates, for example using specific thermal demand values in W/m², while explicitly acknowledging that these are preliminary assumptions. At a later stage, the same modelling structure can be reused with more accurate building-side inputs, without changing the overall logic of the implementation.

Therefore, the purpose of the following code is not to define a single final plant size, but rather to provide a flexible framework in which:

- the building model supplies the thermal demand level,
- the heat pump model is scaled to match the required capacity,
- the TES model is sized consistently with the intended role of the storage, such as buffer operation or energy shifting.

Under this approach, once the building-side assumptions are refined, the HP and TES representations can be updated accordingly with minimal changes to the code structure.


## Possible pre-control optimization targets

Before defining the predictive control strategy itself, it is useful to identify a set of design and operating variables that can be optimized upstream. These variables influence not only the achievable flexibility of the system, but also the efficiency, robustness, and practical usefulness of the controller.

For the **thermal energy storage (TES)**, the most relevant pre-control optimization targets include:

- **Tank size / storage volume**: this directly affects the amount of energy that can be stored, the duration of possible load shifting, and the system ability to reduce short cycling.
- **Usable temperature lift (ΔT)**: this determines the effective storage density and strongly affects the trade-off between compactness of the tank and heat pump efficiency.
- **Tank role**: whether the storage is intended mainly as a **buffer tank** or as an actual **energy-shifting TES** should be clarified before controller design.
- **Stratification level / modelling detail**: the choice between a one-node, two-node, or multi-node representation affects both physical realism and controller complexity.
- **DHW integration**: whether the tank is also used for domestic hot water production introduces additional constraints, including higher temperature requirements and hygienic considerations.
- **Charge/discharge power limits**: these define how fast useful energy can be stored or extracted, and therefore set practical limits on flexibility.
- **Thermal loss coefficient**: the insulation level of the storage affects how valuable long-duration storage actually is.

For the **heat pump (HP)**, relevant upstream optimization targets include:

- **Nominal capacity**: the HP should be sized consistently with the building-side thermal demand and with the intended role of TES.
- **Minimum modulation level**: this is important because low-load operation and part-load behavior strongly affect the need for a buffer volume.
- **Supply temperature level**: this has a direct impact on COP and also constrains the usable ΔT of the tank.
- **Operating mode configuration**: the decision between heating-only, cooling-only, or reversible operation changes the structure of the optimization problem.
- **DHW priority or exclusion**: if DHW is handled by the same unit, the controller may need to manage competing objectives between space heating and sanitary hot water production.
- **Defrost-related constraints**: for air-source heat pumps, operating margins and buffer support during defrost may need to be considered already at the design-logic stage.
- **On/off versus variable-speed operation**: this determines how much flexibility is delegated to the machine itself and how much must instead be absorbed by TES.
- **Part-load efficiency map**: even in simplified models, the dependence of efficiency on load and temperature levels can be important for meaningful optimization.

At system level, some additional variables may also be optimized before the control layer is finalized:

- **Control horizon and time resolution**
- **Comfort bounds or temperature setpoint bands**
- **Electricity price signal or PV self-consumption objective**
- **Relative weighting between efficiency, flexibility, and comfort**

In the following, the implementation is intended to remain flexible with respect to these choices, so that the same modelling structure can be adapted to different assumptions on building demand, tank configuration, and heat pump sizing.
